In [1]:
import numpy as np
import sklearn
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Level 1

In [2]:
ds = load_iris()

In [3]:
ds.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [4]:
ds.data.shape

(150, 4)

In [5]:
ds.target.shape

(150,)

In [6]:
X = ds.data
y = ds.target

X.shape, y.shape

((150, 4), (150,))

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2023)

In [8]:
X_train.shape

(120, 4)

In [9]:
y_test.shape

(30,)

In [10]:
model = SVC()
model.fit(X_train, y_train)

SVC()

In [11]:
y_pred = model.predict(X_test)
y_pred

array([2, 1, 1, 2, 1, 2, 1, 1, 0, 1, 0, 1, 0, 2, 0, 2, 0, 1, 0, 0, 1, 0,
       2, 1, 0, 0, 0, 2, 1, 0])

In [12]:
confusion_matrix(y_test, y_pred)

array([[12,  0,  0],
       [ 0, 11,  0],
       [ 0,  0,  7]])

In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        11
           2       1.00      1.00      1.00         7

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



# Level 2 - hyperparameter tuning
**Pipeline:** X $\rightarrow$ Transform $\rightarrow$ Feature Selection $\rightarrow$ ML $\rightarrow$ y  
- **Transform:** MinMaxScaler, StandardScaler  
- **Feature Selection:** PCA, LDA  
- **ML**: Logistic Regression, SVM, Decision Tree, RandomForest, Naive Bayes  
Bao quát: Lựa chọn kỹ thuật, lựa chọn siêu tham số.


In [14]:
# define utilities
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import itertools
from sklearn.decomposition import PCA


def generate_pipes(pipe_config):
    pipe_config = [list(step.items()) for _, step in pipe_config]
    pipe_config = list(itertools.product(*pipe_config))
    pipe_names = list(map(lambda steps: [name for name, _ in steps], pipe_config))
    pipe_names = list(map(lambda l: ">".join(l), pipe_names))
    pipes = [Pipeline(cfg) for cfg in pipe_config]
    return dict(zip(pipe_names, pipes))


def find_hyperparams(pipe_mapper):
    print("Start the tuning process ...")
    best_pipes = {}
    results = []
    for idx, pipe_name in enumerate(list(pipe_mapper.keys())):
        print(f"{idx + 1}. Tuning pipe: {pipe_name}")

        # Select parameters related to the current pipeline:
        param_grid = {}
        for step_name in pipe_name.split('>'):
            for param_name in parameters.keys():
                if param_name.startswith(step_name):
                    param_grid[param_name] = parameters[param_name]

        # Create a finder and search for the best parameters
        pipe = pipe_mapper[pipe_name]
        finder = GridSearchCV(pipe, param_grid=param_grid, cv=3,
                              scoring="accuracy",
                              refit=True)

        finder.fit(X_train, y_train)
        print("\t best-params: {:>15s}".format(str(finder.best_params_)))
        print(f"\t best-score (accuracy): {finder.best_score_:15.2f}")
        print()

        # Store best pipe
        best_pipes[pipe_name] = finder.best_estimator_
                # Add results to a dataframe
        rs_item = {"Method": pipe_name, "Accuracy": finder.best_score_}
        for key, value in finder.best_params_.items():
            rs_item[key] = value
        results.append(rs_item)

    print("The tuning is done!")
    tuned_table = pd.DataFrame(results)
    tuned_table.set_index("Method")
    return tuned_table, best_pipes

In [15]:
# define pipeline
# Define methods for scaling features
scalers = {
    "MinMaxScaler": MinMaxScaler(),
    "MaxAbsScaler": MaxAbsScaler(),
    "StandardScaler": StandardScaler()
}

reducers = {
    "PCA": PCA()
}


# Define methods for classifying samples
classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=3000),
    "MLPClassifier": MLPClassifier(max_iter=1000),
    "SVC": SVC(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "RandomForestClassifier": RandomForestClassifier()
}

pipe_steps = [
    ('scaler', scalers),
    ('reducers', reducers),
    ('classifier', classifiers)
]

# Define parameters for each method
parameters = {
    "MinMaxScaler__feature_range": [(0, 1), (-1, 1)],
    "StandardScaler__with_mean": [True, False],
    "StandardScaler__with_std": [True, False],
    "SVC__kernel": ['rbf', 'linear'],
    "MLPClassifier__hidden_layer_sizes": [(50,), (100,)],
    "MLPClassifier__activation": ['relu'],
    "DecisionTreeClassifier__criterion": ['entropy'],
    "RandomForestClassifier__n_estimators": [50, 100],
    "RandomForestClassifier__criterion": ['entropy'],
    "PCA__n_components": [0.90, 0.95]
}

In [16]:
# tune hyperparameters
pipe_mapper = generate_pipes(pipe_steps)
tuned_table, best_pipes = find_hyperparams(pipe_mapper)

Start the tuning process ...
1. Tuning pipe: MinMaxScaler>PCA>LogisticRegression
	 best-params: {'MinMaxScaler__feature_range': (-1, 1), 'PCA__n_components': 0.9}
	 best-score (accuracy):            0.94

2. Tuning pipe: MinMaxScaler>PCA>MLPClassifier
	 best-params: {'MLPClassifier__activation': 'relu', 'MLPClassifier__hidden_layer_sizes': (50,), 'MinMaxScaler__feature_range': (-1, 1), 'PCA__n_components': 0.9}
	 best-score (accuracy):            0.95

3. Tuning pipe: MinMaxScaler>PCA>SVC
	 best-params: {'MinMaxScaler__feature_range': (0, 1), 'PCA__n_components': 0.9, 'SVC__kernel': 'linear'}
	 best-score (accuracy):            0.97

4. Tuning pipe: MinMaxScaler>PCA>DecisionTreeClassifier
	 best-params: {'DecisionTreeClassifier__criterion': 'entropy', 'MinMaxScaler__feature_range': (0, 1), 'PCA__n_components': 0.9}
	 best-score (accuracy):            0.94

5. Tuning pipe: MinMaxScaler>PCA>RandomForestClassifier
	 best-params: {'MinMaxScaler__feature_range': (0, 1), 'PCA__n_components':

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perce

	 best-params: {'MLPClassifier__activation': 'relu', 'MLPClassifier__hidden_layer_sizes': (50,), 'PCA__n_components': 0.95}
	 best-score (accuracy):            0.96

8. Tuning pipe: MaxAbsScaler>PCA>SVC
	 best-params: {'PCA__n_components': 0.9, 'SVC__kernel': 'rbf'}
	 best-score (accuracy):            0.95

9. Tuning pipe: MaxAbsScaler>PCA>DecisionTreeClassifier
	 best-params: {'DecisionTreeClassifier__criterion': 'entropy', 'PCA__n_components': 0.95}
	 best-score (accuracy):            0.94

10. Tuning pipe: MaxAbsScaler>PCA>RandomForestClassifier
	 best-params: {'PCA__n_components': 0.95, 'RandomForestClassifier__criterion': 'entropy', 'RandomForestClassifier__n_estimators': 50}
	 best-score (accuracy):            0.94

11. Tuning pipe: StandardScaler>PCA>LogisticRegression
	 best-params: {'PCA__n_components': 0.95, 'StandardScaler__with_mean': True, 'StandardScaler__with_std': False}
	 best-score (accuracy):            0.93

12. Tuning pipe: StandardScaler>PCA>MLPClassifier
	 best-p

In [18]:
tuned_table.to_csv("tuned_results.csv", sep=";")

In [23]:
tuned_table[["Method", "Accuracy", "PCA__n_components"]]

,Method,Accuracy,PCA__n_components
0,MinMaxScaler>PCA>LogisticRegression,0.941667,0.90
1,MinMaxScaler>PCA>MLPClassifier,0.950000,0.90
2,MinMaxScaler>PCA>SVC,0.966667,0.90
3,MinMaxScaler>PCA>DecisionTreeClassifier,0.941667,0.90
4,MinMaxScaler>PCA>RandomForestClassifier,0.950000,0.90
5,MaxAbsScaler>PCA>LogisticRegression,0.941667,0.90
6,MaxAbsScaler>PCA>MLPClassifier,0.958333,0.95
7,MaxAbsScaler>PCA>SVC,0.950000,0.90
8,MaxAbsScaler>PCA>DecisionTreeClassifier,0.941667,0.95
9,MaxAbsScaler>PCA>RandomForestClassifier,0.941667,0.95


In [20]:
# best pipeline
tuned_table[tuned_table["Accuracy"] == tuned_table["Accuracy"].max()].dropna(axis=1)

,Method,Accuracy,MinMaxScaler__feature_range,PCA__n_components,SVC__kernel
2,MinMaxScaler>PCA>SVC,0.966667,"(0, 1)",0.9,linear


In [24]:
#show the selected pipe
selected_pipe_name = tuned_table.Method[tuned_table["Accuracy"].argmax()]
print(f"Name of the best pipe: {selected_pipe_name}")
selected_pipe = best_pipes[selected_pipe_name]

Name of the best pipe: MinMaxScaler>PCA>SVC


In [25]:
from sklearn.metrics import classification_report

print("Start the evaluation process ...")
y_pred = selected_pipe.predict(X_test)
print(classification_report(y_test, y_pred))
print("The evaluation is done!")

Start the evaluation process ...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        11
           2       1.00      1.00      1.00         7

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

The evaluation is done!
